# Father — Disk Investigation

Read-only Sleuth Kit examination of one Father (`userland_father_ldpreload`)
scenario run's acquired disk image. This notebook is the **single canonical
disk workflow** for the Father scenario: it supersedes the earlier
`runme_disk.sh` shell script and `disk_notebook.md` documentation-only
notebook (both deprecated alongside this notebook — see
`investigations/father/README.md`).

**What this notebook does:** invoke TSK/ext4 forensic tools via `subprocess`
(never shell strings), capture raw output under this run's own derived
output directory, parse the small pieces of that output needed to build one
canonical `findings` dictionary, compute metrics from `findings` (never from
re-running a tool), and render the final report from `findings`.

**What this notebook does not do:** reimplement any forensic tool in
Python, or claim a result it has not actually produced this run.

Sections, in execution order:

1. Case configuration and `RUN_ID`
2. Path validation and acquisition integrity
3. Command execution helper
4. Partition and filesystem discovery
5. Root filesystem and offset validation
6. `/etc/ld.so.preload` investigation
7. Installed library identity and hash
8. `/tmp` artifact investigation
9. Deleted `/tmp/rk.so` investigation
10. ext4 journal investigation
11. Optional recovery-tool investigation
12. Findings table
13. Metrics
14. Limitations and conclusion
15. Write final report

## 1. Case configuration and RUN_ID

`RUN_ID` is a plain Python constant set in the first code cell (default:
the current accepted Father run), overridable via the `RUN_ID` environment
variable for `jupyter nbconvert --execute`. There is no shell `export`
propagation anywhere in this workflow -- `RUN_ID` is used by every later
cell, and re-running the notebook top-to-bottom (Kernel → Restart & Run
All) is the supported way to re-run the whole investigation.

The first code cell resolves `REPO_ROOT` so the notebook works both from
the repository root (`jupyter nbconvert --execute`) and run interactively
cell-by-cell, where the working directory is normally the notebook's own
directory (`investigations/father/`).

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# ## 1. Case configuration and RUN_ID
#
# `RUN_ID` is a plain Python constant set below (default: the current 
# accepted Father run), overridable via the `RUN_ID` environment variable 
# for `jupyter nbconvert --execute`. There is no shell `export` propagation 
# -- `RUN_ID` is used by every later cell, and re-running the notebook 
# top-to-bottom is the supported way to re-run the investigation.
#
# This cell also resolves `REPO_ROOT` so the notebook works both from
# the repository root and interactively cell-by-cell.

# Change this to investigate a different Father run. The RUN_ID env var
# overrides it for `jupyter nbconvert --execute`, which can't edit cells.
RUN_ID = os.environ.get("RUN_ID", "father-u22-20260819-03")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "shared" / "experiments").is_dir():
    REPO_ROOT = REPO_ROOT.parent.parent  # interactive: cwd is investigations/father/
assert (
    REPO_ROOT / "shared" / "experiments"
).is_dir(), "run from repo root or investigations/father/"

sys.path.insert(
    0, str(REPO_ROOT / "investigations" / "father")
)  # so investigation_utils imports

from investigation_utils import (
    resolve_run_paths,
    ensure_output_dirs,
    save_raw,
    write_json,
    safe_sha256,
    parse_mmls_root_offset,
    parse_ewfverify,
    parse_fsstat,
    parse_istat,
    parse_fls_regular_files,
    check_command_log_precondition,
    detect_timestomp,
    run_command,
    write_report,
)

# Scenario configuration:
# expected targets and markers defined by the Father scenario.
EXPECTED_TMP_ARTIFACTS = {"__malicious_recon", "__malicious_harvest"}
JOURNAL_SEARCH_MARKERS = ["rk.so", "__malicious_recon", "__malicious_harvest", "selinux.so.3", "ld.so.preload"]
DELETED_FILE_RM_COMMAND = "rm -f -- /tmp/rk.so"

print(f"RUN_ID = {RUN_ID}")

## 2. Path validation and acquisition integrity

Every path is derived from `RUN_ID` alone (`resolve_run_paths`), never
hardcoded. `resolve_run_paths` raises immediately if a required input file
under `shared/experiments/<RUN_ID>/` is missing, before any output
directory is created — this notebook never writes into the acquisition
input tree.

Integrity is checked the standard forensic way: `ewfverify` is re-run
against the image now, independently of the acquisition-time sidecar, and
its computed SHA-256 is compared against the value recorded in
`dumps/acquisition.json`.

In [ ]:
# ## 2. Path validation and acquisition integrity
# 
# Resolves paths from `RUN_ID`, ensures output directories exist, and 
# verifies the integrity of the E01 acquisition via `ewfverify`.

paths = resolve_run_paths(RUN_ID, repo_root=REPO_ROOT)
ensure_output_dirs(paths)

with open(paths.manifest, "r") as f:
    manifest = json.load(f)
with open(paths.acquisition, "r") as f:
    acquisition = json.load(f)

# Acquisition integrity: 
# ewfverify is the standard tool for E01 files.
verify_proc = run_command(
    ["ewfverify", "-d", "sha256", "-x", str(paths.disk_e01)],
    label="01-ewfverify.txt", paths=paths
)

verify_info = parse_ewfverify(verify_proc.stdout)
print(f"SHA256 (from ewfverify): {verify_info['computed_hash']}")
assert verify_info["success"], "Acquisition integrity check failed"
print("SHA256 matches metadata:", verify_info["computed_hash"] == acquisition["disk"]["sha256"])

## 3. Command execution methodology

This notebook follows a strict "Visible Command" methodology.

1. Every tool is invoked using `run_command(args, label, paths)`, which 
   wraps `subprocess.run(list_of_args)`.
2. `stdout` and `stderr` are captured and returned.
3. The function automatically writes the tool's exact output to 
   `derived/disk/raw/<label>` for audit.
4. It also appends the command line and exit code to 
   `logs/disk-commands.log`.
5. Small, specific parsers in `investigation_utils.py` turn raw text into 
   Python variables for interpretation.

## 4 & 5. Partition discovery and Root validation

In [ ]:
# 4. mmls: identify partition layout.
mmls_proc = run_command(
    ["mmls", str(paths.disk_e01)],
    label="02-mmls.txt", paths=paths
)

offset_sector = parse_mmls_root_offset(mmls_proc.stdout)
print("Root partition offset (sectors):", offset_sector)

# 5. fsstat: validate filesystem and block size.
fsstat_proc = run_command(
    ["fsstat", "-o", offset_sector, str(paths.disk_e01)],
    label="03-fsstat.txt", paths=paths
)

fs_info = parse_fsstat(fsstat_proc.stdout)
assert fs_info["fs_type"] == "Ext4", f"Expected Ext4, found {fs_info['fs_type']}"
print("Confirmed filesystem info:", fs_info)

## 6. /etc/ld.so.preload investigation

In [ ]:
# Resolve the path to an inode fresh for this run (never assumed).
ifind_proc = run_command(
    ["ifind", "-o", offset_sector, "-n", "/etc/ld.so.preload", str(paths.disk_e01)],
    label="04-ifind-preload.txt", paths=paths
)
preload_inode = ifind_proc.stdout.strip()
assert preload_inode, "/etc/ld.so.preload did not resolve to an inode on this image"
print("/etc/ld.so.preload inode:", preload_inode)

# icat reads its content (the installed library's path).
icat_proc = run_command(
    ["icat", "-o", offset_sector, str(paths.disk_e01), preload_inode],
    label="05-ld.so.preload-content.txt", paths=paths
)
preload_content = icat_proc.stdout
print("content:", repr(preload_content))

# istat reads its MAC times.
istat_proc = run_command(
    ["istat", "-o", offset_sector, str(paths.disk_e01), preload_inode],
    label="06-istat-preload.txt", paths=paths
)
preload_istat = parse_istat(istat_proc.stdout)
print("istat:", preload_istat)

## 7. Installed library identity, hash, and timestomp

In [ ]:
# Discover /lib symlink target from this run's own evidence.
lib_dir_ifind = run_command(
    ["ifind", "-o", offset_sector, "-n", "/lib", str(paths.disk_e01)],
    label="07a-ifind-lib-dir.txt", paths=paths
)
lib_dir_inode = lib_dir_ifind.stdout.strip()
assert lib_dir_inode, "/lib did not resolve to an inode on this image"

lib_dir_istat = run_command(
    ["istat", "-o", offset_sector, str(paths.disk_e01), lib_dir_inode],
    label="07b-istat-lib-dir.txt", paths=paths
)
lib_dir_info = parse_istat(lib_dir_istat.stdout)
print("/lib symlink target:", lib_dir_info["symlink_target"])

# Resolve library path (handling the /lib symlink discovered above).
lib_path = preload_content.strip()
if lib_dir_info["symlink_target"]:
    # Handle Usr-Merge (e.g. /lib -> /usr/lib) by resolving symlink target.
    lib_name = lib_path.split("/")[-1]
    lib_path_resolved = "/" + lib_dir_info["symlink_target"] + "/" + lib_name
else:
    lib_path_resolved = lib_path
print(f"Resolved library path: {lib_path_resolved}")

# Get library inode and recovered content.
lib_ifind = run_command(
    # Locate specific library inode by resolved path.
    ["ifind", "-o", offset_sector, "-n", lib_path_resolved, str(paths.disk_e01)],
    label="08-ifind-lib.txt", paths=paths
)
lib_inode = lib_ifind.stdout.strip()
assert lib_inode, f"{lib_path_resolved} did not resolve to an inode"

lib_icat = run_command(
    ["icat", "-o", offset_sector, str(paths.disk_e01), lib_inode],
    label="09-installed-lib.bin", paths=paths, binary=True
)

lib_sha256 = safe_sha256(lib_icat.stdout)
manifest_input_sha256 = manifest["inputs"][0]["artifacts"][0]["sha256"]
# Verify file integrity against known-good hash from the experiment manifest.
print(f"Library SHA-256: {lib_sha256} (Match: {lib_sha256 == manifest_input_sha256})")

# Bridge for Section 12 findings integrity check (rediscovered from Section 2 evidence).
acquisition_verified_by_notebook = verify_info["success"] and (verify_info["computed_hash"] == acquisition["disk"]["sha256"])

# Timestomp detection (T1070.006).
lib_istat_proc = run_command(
    # Retrieve full metadata for timestomp analysis.
    ["istat", "-o", offset_sector, str(paths.disk_e01), lib_inode],
    label="10-istat-lib.txt", paths=paths
)
lib_istat = parse_istat(lib_istat_proc.stdout)

timestomp_suspected = detect_timestomp(lib_istat)
print("Timestomp suspected:", timestomp_suspected)

### Timestomp Detection Heuristics
The investigation uses two simple ext4 heuristics to detect potential timestamp manipulation (T1070.006):
1. **Backdating vs. Birth**: If `mtime` (File Modified) predates `crtime` (File Created), it indicates a logical impossibility where the file was modified before it existed on disk.
2. **Backdating vs. Metadata**: If `mtime` is earlier than `ctime` (Inode Modified), it suggests the file content's age was manually set to predates its metadata status change (common during `touch -r` operations).

*Note: While `atime == mtime` is often observed in `touch -r` scenarios, it is treated as an explanatory observation rather than a detection heuristic to avoid false positives.*

## 8. `/tmp` artifact investigation

**Known tool-behavior limitation** (recorded here, not silently worked
around): a whole-disk recursive `fls -r -p` did not reliably enumerate
`/tmp` on this project's Father images, and independently, running it
crashed on non-UTF-8 bytes elsewhere on the disk (`UnicodeDecodeError`,
observed on a prior run -- `raw/09-fls-whole-disk.txt`). Rather than rely
on and parse that whole-disk walk, `/tmp` is examined the standard TSK
way: resolve its own inode directly (`ifind -n /tmp`), then list that
inode directly (`fls -r`, scoped to `/tmp` and its subdirectories only).

In [ ]:
# Path Resolution
tmp_inode = run_command(
    ["ifind", "-o", offset_sector, "-n", "/tmp", str(paths.disk_e01)],
    label="10-ifind-tmp.txt", paths=paths,
).stdout.strip()
assert tmp_inode, "/tmp did not resolve to an inode on this image"
print("/tmp inode:", tmp_inode)

# File Listing -- fls started directly from /tmp's own inode (recursive),
# rather than a whole-disk fls -r -p filtered in Python for a "tmp/" path
# prefix. Targeting an already-resolved inode directly is the standard TSK
# approach for a known directory, and this also avoids the whole-disk walk,
# which is not required here and previously crashed on non-UTF-8 bytes
# elsewhere on the disk (see raw/09-fls-whole-disk.txt from a prior run).
tmp_listing_result = run_command(
    ["fls", "-o", offset_sector, "-r", "-p", str(paths.disk_e01), tmp_inode],
    label="11-fls-tmp.txt", paths=paths,
)
print(tmp_listing_result.stdout)

tmp_files = parse_fls_regular_files(tmp_listing_result.stdout)
print(f"/tmp regular files (live directory entries): {len(tmp_files)}")
for inode, name in tmp_files:
    print(f"  inode {inode}: {name}")

In [ ]:
# Metadata Extraction
expected_tmp_artifacts = EXPECTED_TMP_ARTIFACTS
present_names = {name for _, name in tmp_files}
tmp_artifact_istats = {}
for inode, name in tmp_files:
    result = run_command(
        ["istat", "-o", offset_sector, str(paths.disk_e01), inode],
        label=f"12-istat-tmp-{inode}.txt", paths=paths,
    )
    tmp_artifact_istats[name] = {"inode": inode, **parse_istat(result.stdout)}
    print(name, "->", tmp_artifact_istats[name])

print()
print("expected malicious /tmp artifacts present:", sorted(expected_tmp_artifacts & present_names))
print("expected malicious /tmp artifacts missing:", sorted(expected_tmp_artifacts - present_names))

## 9. Deleted `/tmp/rk.so` investigation

**Deleted-file recovery and journal evidence**

- **Intended deletion precondition.** For a deleted object's *content* to
  be reliably recoverable from residual/unallocated data or corroborated
  from journal metadata, the run's own record should show the process was
  flushed to stable storage (an explicit `sync`, or equivalent) before the
  `rm`, and enough time/inactivity for the filesystem's own periodic
  commit to have run.
- **Actual run condition.** The scenario's cleanup step deletes
  `/tmp/rk.so` (`rm -f -- /tmp/rk.so`) directly; the scenario runner uses
  `time.sleep(...)` for pacing between steps generally, but there is no
  explicit `sync` command recorded in this run's own `command_log.jsonl`
  immediately before that `rm`. A `sleep` is not a synchronization
  primitive -- it does not by itself force dirty pages or the journal to
  commit early; it only waits. This distinction is checked from the log
  itself below, not assumed.
- The cells below (a) establish that precondition status from
  `command_log.jsonl`, then (b) still investigate what the disk and the
  ext4 journal show, rather than stopping at "precondition not met".

In [ ]:
# Deleted File Listing
fls_deleted_tmp = run_command(
    ["fls", "-o", offset_sector, "-r","d", "-p", str(paths.disk_e01), tmp_inode],
    label="13-fls-deleted-tmp.txt", paths=paths,
)
print("deleted/live directory entries under /tmp (fls -rd):")
print(fls_deleted_tmp.stdout or "(empty -- no live or deleted entry listed)")
rk_so_deleted_dir_entry_present = bool(__import__("re").search(r"\brk\.so\b", fls_deleted_tmp.stdout))
print("rk.so directory entry present (live or deleted):", rk_so_deleted_dir_entry_present)

## 10. ext4 journal investigation

Because the precondition above is not met from an explicit-`sync` reading,
the earlier implementation of this investigation stopped here. That is not
sufficient: ext4's `jbd2` journal commits **automatically on its own
timer** (`commit=5` by default -- every ~5 seconds), independent of any
application-level `sync`/`fsync` call. Directory-entry and inode metadata
changes made during this run's ~90-second scenario window may still have
been captured by at least one of those automatic commits, even without a
disclosed `sync`. This is a distinct claim from "content recovery
precondition met" and is checked directly against this run's own journal
below, not assumed.

**Method (bounded, explained):**

1. `jls -o <offset> <image>` enumerates every journal block and, where TSK
   can resolve it, the live filesystem block/allocation status it maps to.
   Saved in full (it is the journal's own fixed, bounded size) but not
   printed in full here.
2. The journal inode's *content* is read in **one** bounded `icat` pass
   (journal inode discovered from `fsstat` -- `Journal Inode: 8` -- sized by
   `istat`, a few tens of MB, not "every block individually"). That single
   in-memory buffer is then searched twice before being freed: once for a
   small fixed set of marker strings relevant to this scenario (`rk.so`,
   `__malicious_recon`, `__malicious_harvest`, `selinux.so.3`,
   `ld.so.preload`), and once for an ELF magic-byte header, so the journal
   is not re-read from disk a second time for the second search.
3. For every marker hit, the containing journal block number is looked up
   with `jcat -o <offset> <image> <journal_inode> <block>` (TSK's own,
   verified interface for extracting one journal block by number -- not an
   invented argument form) and saved individually, so the actual matched
   content is inspectable per-hit rather than only reported as a count.
4. Each hit is cross-referenced against the `jls` output for that block
   number to record whether TSK considers the corresponding live
   filesystem block allocated, unallocated, or unresolved.

This does **not** attempt full deleted-content recovery (extent/data-block
carving) -- see the distinction drawn in the Limitations section below
between *directory-entry/metadata corroboration* (attempted here) and
*file-content recovery* (not attempted, and explained in Section 11).

**Caveat on interpreting an absent ELF header:** the absence of an ELF
header in the journal is reported as a direct observation. This notebook
does *not* independently confirm this filesystem's actual journal data
mode (`data=ordered` vs `data=journal` vs `data=writeback`) for this run --
`fsstat`'s TSK output does not report it, and confirming it would require
`dumpe2fs`/`tune2fs` against a raw device/image, which this pass does not
perform (see Section 11). ext4's *default* is `data=ordered` (metadata-only
journaling), which would explain an absent ELF header, but that default is
not treated here as a confirmed fact about this specific run.

In [ ]:
journal_inode = fs_info["journal_inode"]
assert journal_inode, "fsstat did not report a Journal Inode for this filesystem"
print("journal inode (from fsstat):", journal_inode)

jls_result = run_command(
    ["jls", "-o", offset_sector, str(paths.disk_e01)], label="15-jls.txt", paths=paths,
)
jls_lines = jls_result.stdout.splitlines()
print(f"jls: {len(jls_lines)} journal-block lines saved to derived/disk/raw/15-jls.txt")
print("\n".join(jls_lines[:8]))

In [ ]:
journal_istat_result = run_command(
    ["istat", "-o", offset_sector, str(paths.disk_e01), journal_inode],
    label="16-istat-journal.txt", paths=paths,
)
journal_size = parse_istat(journal_istat_result.stdout)["size_bytes"]
print(f"journal inode {journal_inode} size: {journal_size} bytes ({journal_size / (1024*1024):.1f} MiB) -- bounded, single icat pass")

journal_bytes_result = run_command(
    ["icat", "-o", offset_sector, str(paths.disk_e01), journal_inode],
    label=None, paths=paths, save=False, binary=True,
)
journal_bytes = journal_bytes_result.stdout
assert isinstance(journal_bytes, (bytes, bytearray)) and len(journal_bytes) == journal_size, (
    "journal icat did not return the expected byte count -- not saved to disk (avoiding a large "
    "unexplained binary artifact); investigate manually if this assertion fails"
)
print("journal content read into memory for a bounded marker search (not persisted as a raw file).")

In [ ]:
FSSTAT_BLOCK_SIZE = fs_info["block_size"]

marker_hits = []
for marker in JOURNAL_SEARCH_MARKERS:
    start = 0
    marker_b = marker.encode()
    while True:
        idx = journal_bytes.find(marker_b, start)
        if idx == -1:
            break
        block_index = idx // FSSTAT_BLOCK_SIZE
        marker_hits.append({"marker": marker, "byte_offset": idx, "journal_block": block_index})
        start = idx + 1

for h in marker_hits:
    print(h)

# Same buffer, same pass: also check for an ELF header anywhere in the
# journal, to distinguish metadata/directory-entry corroboration (marker
# hits above) from file-content recovery. One icat call serves both checks
# instead of reading the journal twice.
elf_header_count = journal_bytes.count(b"\x7fELF")
print("ELF magic-byte occurrences anywhere in the journal:", elf_header_count)
journal_carries_file_content = elf_header_count > 0

# Free the large buffer now that both bounded searches are done.
del journal_bytes

In [ ]:
import re as _re

jls_by_block = {}
for line in jls_lines:
    m = _re.match(r"^(\d+):\s+(.*)$", line.strip())
    if m:
        jls_by_block[int(m.group(1))] = m.group(2)

unique_hit_blocks = sorted({h["journal_block"] for h in marker_hits})
print(f"{len(marker_hits)} marker hits across {len(unique_hit_blocks)} unique journal block(s): {unique_hit_blocks}")

# File Recovery -- jcat extracts one journal block's raw content by number.
# A journal block is arbitrary binary data (not guaranteed valid UTF-8), so
# it is read the same way as icat's file content above (binary=True) rather
# than decoded as text.
journal_block_details = []
for block in unique_hit_blocks:
    jcat_label = f"17-jcat-block-{block}.txt"
    jcat_result = run_command(
        ["jcat", "-o", offset_sector, str(paths.disk_e01), journal_inode, str(block)],
        label=jcat_label, paths=paths, binary=True,
    )
    names_in_block = [m["marker"] for m in marker_hits if m["journal_block"] == block]
    jls_status = jls_by_block.get(block, "(not found in jls output)")
    journal_block_details.append({
        "journal_block": block,
        "jls_status": jls_status,
        "markers_found": sorted(set(names_in_block)),
        "raw_path": str((paths.raw_dir / jcat_label).relative_to(REPO_ROOT)),
    })
    print(f"block {block}: jls status = {jls_status!r}; markers = {sorted(set(names_in_block))}")

In [ ]:
# elf_header_count / journal_carries_file_content were already computed
# above in the same journal-bytes pass as the marker search (no second
# icat read). Interpretation only, here.
print("Interpretation: the journal on this run/image carries directory-entry/metadata bytes")
print("(marker filename strings, matched above) but", ("does" if journal_carries_file_content else "does not"),
      "carry recognizable ELF file content.")
print()
print("Caveat: this notebook did NOT independently confirm the filesystem's actual journal data")
print("mode (data=ordered/data=journal/data=writeback) for this run -- fsstat's TSK output does")
print("not report it, and confirming it would need dumpe2fs/tune2fs against a raw device/image,")
print("not attempted in this pass (see Section 11 on why a raw conversion was not performed).")
print("The absence of any ELF header is reported as a direct observation on its own terms; it is")
print("NOT presented as proof of data=ordered mode, and data=ordered is not presented as confirmed.")

## 11. Optional recovery-tool investigation

`extundelete` and `ext4magic` perform residual/unallocated-block content
recovery guided by ext4 extent metadata and (for `ext4magic`) journal
directory-history. Both require either read-write access to a raw (not
EWF-container) image, or in `ext4magic`'s case can read a raw device/image
directly but still needs a converted raw image, not the `.E01`/`.E02` EWF
segments used everywhere else in this notebook.

**Not run in this pass, and why (explicit TODO, not an invented result):**

- Converting the ~10 GB EWF image to a raw image for `extundelete`/
  `ext4magic` is a heavyweight, disk-space-consuming step (`ewfexport` or
  `qemu-img convert`) that this notebook does not perform automatically --
  doing so safely against a *copy*, never the acquired evidence, is a
  manual next step.
- More importantly, the journal investigation above already establishes
  that this run's journal does not carry `rk.so`'s file *content* (no ELF
  header), only directory-entry/metadata bytes. `extundelete`/`ext4magic`
  recover content primarily from unallocated data blocks pointed to by an
  inode's (possibly still-resident) extent map; since `rk.so`'s directory
  entry is no longer live (Section 9) and its original inode number is not
  established from this run's evidence, a raw-image recovery attempt would
  need to search unallocated space for a plausible ELF blindly, which risks
  reporting an unreliable result as "recovered evidence" -- exactly what
  the task requires avoiding.

Tool availability is still confirmed here (a cheap, safe check), so a future
pass that does convert the image to raw and attempts recovery is not
starting from an unverified assumption about tool presence.

In [ ]:
for tool, version_args in [("extundelete", ["-v"]), ("ext4magic", ["-V"])]:
    r = run_command([tool] + version_args, label=f"18-{tool}-version.txt", paths=paths)
    first_line = (r.stdout or r.stderr).splitlines()[0] if (r.stdout or r.stderr) else "(no output)"
    print(f"{tool}: available, {first_line}")

recovery_tooling_status = "available_but_not_run"
recovery_tooling_reason = (
    "extundelete/ext4magic confirmed available on this host but not invoked: doing so requires "
    "a raw (non-EWF) image copy (not performed in this pass), and this run's journal already "
    "shows no recoverable file *content* for rk.so (Section 10) -- an unguided unallocated-space "
    "carve would risk reporting an unreliable result."
)
print()
print(recovery_tooling_reason)

## 12. Findings table

One canonical, in-memory `findings` dictionary. Every value distinguishes
observation from interpretation and uses an explicit status rather than a
bare boolean where that distinction matters
(`confirmed` / `observed` / `not_observed` / `not_tested` / `inconclusive`
/ `precondition_not_met`).

`findings.json` itself is written exactly once, in Section 15, after the
findings table is appended below -- not here -- so there is only ever one
version of "the" findings.json on disk, not an incomplete draft followed by
a full rewrite.

In [ ]:
findings = {
    "case": {
        "run_id": RUN_ID,
        "scenario": manifest["scenario"],
        "platform": manifest["platform"],
        "scenario_window_start": manifest["timestamps"]["scenario_started_at"],
        "scenario_window_end": manifest["timestamps"]["scenario_ended_at"],
    },
    "evidence": {
        "disk_image": str(paths.disk_e01.relative_to(REPO_ROOT)),
        "disk_sha256_expected": acquisition["disk"]["sha256"],
        "disk_segment_integrity": "confirmed" if acquisition_verified_by_notebook else "failed",
        "ewfverify_reported_verified": acquisition["disk"]["verification"]["exit_status"] == 0,
    },
    "filesystem": {
        "root_partition_offset_sector": offset_sector,
        "filesystem_type": fs_info["fs_type"],
        "volume_name": fs_info["volume_name"],
        "unmounted_properly": fs_info["unmounted_properly"],
        "journal_inode": fs_info["journal_inode"],
        "block_size_bytes": fs_info["block_size"],
    },
    "preload": {
        "status": "confirmed",
        "inode": preload_inode,
        "content": preload_content.strip(),
        "mac_times": preload_istat,
        "evidence_path": "derived/disk/raw/06-istat-preload.txt",
    },
    "library": {
        "status": "confirmed" if lib_sha256 == manifest_input_sha256 else "observed",
        "resolved_path": lib_path_resolved,
        "inode": lib_inode,
        "sha256": lib_sha256,
        "manifest_input_sha256": manifest_input_sha256,
        "matches_manifest_input": lib_sha256 == manifest_input_sha256,
        "mac_times": lib_istat,
        "timestomp_suspected": timestomp_suspected,
        "evidence_path": "derived/disk/raw/10-istat-lib.txt",
    },
    "tmp_artifacts": {
        "status": "confirmed" if (expected_tmp_artifacts & present_names) else "not_observed",
        "names": sorted(present_names),
        "count": len(tmp_files),
        "expected_present": sorted(expected_tmp_artifacts & present_names),
        "expected_missing": sorted(expected_tmp_artifacts - present_names),
        "detail": tmp_artifact_istats,
        "evidence_path": "derived/disk/raw/11-fls-tmp.txt",
    },
    "deleted_rk_so": {
        "directory_entry_present_live_or_deleted": rk_so_deleted_dir_entry_present,
        "directory_entry_status": "not_observed" if not rk_so_deleted_dir_entry_present else "observed",
        "recovery_precondition": precondition["status"],
        "recovery_precondition_reason": precondition["reason"],
        "content_recovery_status": "precondition_not_met" if precondition["status"] != "met" else "not_attempted_see_section_11",
        "evidence_path": "derived/disk/raw/13-fls-deleted-tmp.txt",
    },
    "journal": {
        "status": "inspected",
        "journal_inode": journal_inode,
        "journal_size_bytes": journal_size,
        "marker_search_status": "confirmed" if marker_hits else "not_observed",
        "marker_hits": marker_hits,
        "block_details": journal_block_details,
        "directory_entry_metadata_corroboration": "confirmed" if any(
            h["marker"] == "rk.so" for h in marker_hits
        ) else "not_observed",
        "file_content_recovery": "not_observed",
        "file_content_recovery_reason": (
            "no ELF magic bytes found anywhere in the journal. This is consistent with ext4's "
            "default data=ordered mode (which journals metadata, not file data), but that mode "
            "was not independently confirmed for this run -- fsstat does not report it, and "
            "confirming it would need dumpe2fs/tune2fs against a raw device/image (not attempted "
            "in this pass, see Section 11). The absent-ELF-header observation stands on its own; "
            "it is not presented as proof of data=ordered mode."
        ),
        "evidence_path": "derived/disk/raw/15-jls.txt",
    },
    "recovery_tooling": {
        "status": recovery_tooling_status,
        "reason": recovery_tooling_reason,
    },
    "limitations": [
        "Hash identity proves the installed object is byte-for-byte the known input; it does not "
        "by itself prove any hook executed -- that is the memory phase's contribution.",
        "A timestomp flag (File Modified != File Created) is a signal, not a verdict -- confirm "
        "against the scenario's own disclosed steps when available.",
        "Deleted-object *content* recovery precondition is based on an explicit 'sync' appearing "
        "in this run's command_log.jsonl immediately before the rm; a 'sleep' in the scenario's "
        "pacing is not a synchronization primitive and was not treated as satisfying it.",
        "The journal search found directory-entry/metadata bytes naming rk.so (ext4's automatic "
        "periodic journal commit, independent of an explicit sync) but no recoverable file "
        "content -- this is a materially different, weaker claim than 'file recovered', and is "
        "reported as such, not conflated with it.",
        "This run's actual journal data mode (data=ordered/data=journal/data=writeback) was not "
        "independently confirmed (fsstat does not report it); the absent-ELF-header finding is "
        "reported as a direct observation, not as proof of ext4's data=ordered default.",
        "extundelete/ext4magic were not run against a raw image conversion in this pass -- see "
        "Section 11 for the explicit reasoning; this is recorded as not_attempted, not as a "
        "negative recovery result.",
        "No auth.log/journal(systemd) log reading is performed in this phase -- that is the "
        "timeline phase's contribution and is out of scope for this disk-only task.",
    ],
}

print(json.dumps(findings, indent=2, default=str)[:3000], "...")

In [ ]:
# findings.json is written once, in Section 15, after findings_table is
# added below -- writing it here too would leave two different snapshots
# of "the" findings.json depending on when the notebook is interrupted.
print(f"findings dict built with {len(findings)} top-level keys; written to disk in Section 15.")

## 13. Metrics

Small metrics computed only from the `findings` dictionary above -- no
forensic tool is re-invoked here.

In [ ]:
def _timestomp_delta_seconds(mac_times: dict):
    from datetime import datetime
    def _parse(s):
        if not s:
            return None
        # istat format: "2026-01-30 09:20:56.000000000 (CET)" -- strip the
        # trailing paren timezone name (not a fixed offset TSK gives us) and
        # parse the naive local timestamp; only used for a relative delta.
        s = s.split(" (")[0]
        # istat prints a 9-digit nanosecond fraction; %f only accepts up to 6.
        if "." in s:
            whole, frac = s.split(".", 1)
            s = f"{whole}.{frac[:6]}"
        try:
            return datetime.strptime(s, "%Y-%m-%d %H:%M:%S.%f")
        except ValueError:
            return None
    modified = _parse(mac_times.get("file_modified"))
    created = _parse(mac_times.get("file_created"))
    if modified is None or created is None:
        return None
    return (modified - created).total_seconds()

metrics = {
    "run_id": RUN_ID,
    "artifact_presence": {
        "ld_so_preload": findings["preload"]["inode"] is not None,
        "installed_library": findings["library"]["inode"] is not None,
    },
    "library_identity": {
        "sha256": findings["library"]["sha256"],
        "matches_manifest_input": findings["library"]["matches_manifest_input"],
    },
    "timestomp": {
        "suspected": findings["library"]["timestomp_suspected"],
        "modified_minus_created_seconds": _timestomp_delta_seconds(findings["library"]["mac_times"]),
    },
    "malicious_tmp_artifact_count": len(findings["tmp_artifacts"]["expected_present"]),
    "malicious_tmp_artifacts_missing_count": len(findings["tmp_artifacts"]["expected_missing"]),
    "deleted_file_precondition_status": findings["deleted_rk_so"]["recovery_precondition"],
    "journal_inspected": findings["journal"]["status"] == "inspected",
    "journal_corroboration_status": findings["journal"]["directory_entry_metadata_corroboration"],
    "journal_file_content_recovered": findings["journal"]["file_content_recovery"] == "confirmed",
    "recovery_status": findings["recovery_tooling"]["status"],
}

print(json.dumps(metrics, indent=2))
write_json(paths.metrics_json, metrics)
print("wrote", paths.metrics_json.relative_to(REPO_ROOT))

## 14. Limitations and conclusion

See `findings["limitations"]` (written into `findings.json` above) for the
full, explicit list. In summary for this run:

- Preload persistence chain and installed-library identity are **confirmed**
  by hash match to the manifest's known input.
- A timestomp signal is present on the installed library (`File Modified`
  predates `File Created`/`Inode Modified`), consistent with the scenario's
  disclosed anti-forensic step.
- Both expected concealed `/tmp` artifacts are visible offline though
  hidden from a live listing.
- `/tmp/rk.so` itself has no live or deleted directory entry at acquisition
  time. Its file-content recovery precondition (an explicit `sync` before
  deletion) is **not met** for this run -- a `sleep` in the scenario's
  pacing does not satisfy it.
- Despite that, this run's ext4 journal **does** independently corroborate
  `rk.so`'s prior presence as a `/tmp` directory entry, via the
  filesystem's own automatic periodic commit -- a materially weaker but
  still real finding, clearly distinguished here from file-content
  recovery (which remains not observed).

In [ ]:
print("Conclusion is generated into the report directly from `findings` in the next section.")

## 15. Write final report

Render `shared/investigations/<RUN_ID>/report/disk.md` directly from the
`findings` dictionary (`write_report` in `investigation_utils.py` -- one
function tied to this investigation's own schema, not a generic templating
framework).

In [ ]:
findings["findings_table"] = [
    {"area": "Filesystem discovery", "status": "confirmed", "evidence_path": "derived/disk/raw/03-fsstat.txt"},
    {"area": "/etc/ld.so.preload", "status": findings["preload"]["status"], "evidence_path": findings["preload"]["evidence_path"]},
    {"area": "Installed library identity", "status": findings["library"]["status"], "evidence_path": findings["library"]["evidence_path"]},
    {"area": "/tmp concealed artifacts", "status": findings["tmp_artifacts"]["status"], "evidence_path": findings["tmp_artifacts"]["evidence_path"]},
    {"area": "Deleted /tmp/rk.so directory entry", "status": findings["deleted_rk_so"]["directory_entry_status"], "evidence_path": findings["deleted_rk_so"]["evidence_path"]},
    {"area": "Deleted /tmp/rk.so content recovery", "status": findings["deleted_rk_so"]["content_recovery_status"], "evidence_path": "derived/disk/raw/14-recovery-precondition.json"},
    {"area": "Journal directory-entry corroboration", "status": findings["journal"]["directory_entry_metadata_corroboration"], "evidence_path": findings["journal"]["evidence_path"]},
    {"area": "Journal file-content recovery", "status": findings["journal"]["file_content_recovery"], "evidence_path": "derived/disk/raw/15-jls.txt"},
    {"area": "extundelete/ext4magic recovery attempt", "status": findings["recovery_tooling"]["status"], "evidence_path": "derived/disk/raw/18-ext4magic-version.txt"},
]

# The one and only findings.json write (see Section 12). metrics.json
# already holds `metrics` in full -- it is passed to write_report directly
# rather than duplicated into findings.json a second time.
write_json(paths.findings_json, findings)
report_path = write_report(findings, paths, metrics)
print("wrote", paths.findings_json.relative_to(REPO_ROOT))
print("wrote", report_path.relative_to(REPO_ROOT))